# 📘 Semaine 3 — GPIO (HAL / LL / Registres)

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 cours + 1h30 atelier + 1h30 homework)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs pédagogiques de la semaine

À la fin de cette semaine, l'étudiant sera capable de :

1. **Décrire** l'organisation des ports GPIO (A, B, C) et leurs registres.
2. **Configurer** un GPIO en entrée ou sortie (push-pull, open-drain, pull-up, pull-down).
3. **Programmer** un GPIO avec **HAL**, **LL** et en **accès direct aux registres**.
4. **Lire** un bouton et piloter une LED avec anti-rebond.
5. **Choisir** la bonne méthode (HAL / LL / registres) selon le contexte.

---

## 🗺️ Plan de la semaine

| Partie | Contenu | Durée |
|---|---|---|
| **A — Cours** | Activités 1 à 5 | 1h30 |
| **B — Atelier** | TP3 : LED + bouton + anti-rebond | 1h30 |
| **C — Homework** | Exercices 1 à 3 | 1h30 |
| **D — Auto-évaluation** | Checklist finale | 5 min |

---
# 🎓 PARTIE A — COURS INTÉGRÉ (1h30)

## 🔹 Activité 1 — Rappel & mise en contexte (10 min)

### 🔄 Rappel des semaines 1 et 2
- **Cortex-M3** @ 72 MHz, pipeline 3 étages, NVIC.
- **Bus** : AHB / APB1 (36 MHz) / APB2 (72 MHz).
- **Mappe** : Flash `0x0800_0000`, SRAM `0x2000_0000`, périphériques `0x4000_0000`.
- **Horloges** : HSE ×9 PLL → SYSCLK = 72 MHz.
- **GPIOC** base : `0x4001_1000` — sur bus **APB2**.

### ✍️ Questions flash (2 min)
1. Quelle est l'adresse de base de GPIOA ? → ...
2. Quel bus alimente les GPIO ? → ...
3. Quelle macro active l'horloge de GPIOC ? → ...
4. Combien de GPIO disponibles sur le LQFP48 ? → ...

---

## 🔹 Activité 2 — Organisation des GPIO (20 min)

### 📖 2.1 — Ports disponibles

Le STM32F103C6T6 expose **3 ports GPIO** :

| Port | Adresse de base | Broches (LQFP48) | Horloge |
|---|---|---|---|
| **GPIOA** | `0x4001_0800` | PA0 – PA15 | APB2 |
| **GPIOB** | `0x4001_0C00` | PB0 – PB15 | APB2 |
| **GPIOC** | `0x4001_1000` | PC13 – PC15 | APB2 |

> ⚠️ Sur la Blue Pill, le port C n'expose que **PC13, PC14, PC15** (utilisés aussi pour le quartz LSE).

### 📖 2.2 — 16 broches par port

Chaque port possède **16 broches** numérotées de 0 à 15.
Chaque broche peut être configurée indépendamment :

```
GPIOx
├── Pin 0  → EXTI0, TIM2_CH1, USART2_CTS, ADC_IN0 …
├── Pin 1  → EXTI1, TIM2_CH2, USART2_RTS, ADC_IN1 …
├── Pin 2  → EXTI2, TIM2_CH3, USART2_TX, ADC_IN2 …
├── Pin 3  → EXTI3, TIM2_CH4, USART2_RX, ADC_IN3 …
└── …
```

Chaque broche peut être :
- **Entrée** (analogique, flottante, pull-up, pull-down)
- **Sortie** (push-pull ou open-drain)
- **Fonction alternative** (TIM, USART, SPI, I2C…)

### 📖 2.3 — Registres GPIO (STM32F1)

| Registre | Offset | Rôle |
|---|---|---|
| **CRL** | `0x00` | Configuration des pins 0 à 7 |
| **CRH** | `0x04` | Configuration des pins 8 à 15 |
| **IDR** | `0x08` | Lecture de l'état des entrées |
| **ODR** | `0x0C` | Écriture de l'état des sorties |
| **BSRR** | `0x10` | Set / Reset atomique |
| **BRR** | `0x14` | Reset uniquement (déprécié) |
| **LCKR** | `0x18` | Verrouillage de la configuration |

### 📖 2.4 — Structure des registres CRL / CRH

Chaque pin utilise **4 bits** dans CRL (ou CRH) :

```
Bit   : 31 30 29 28 | 27 26 25 24 | … | 7 6 5 4 | 3 2 1 0
Pin   :    Pin7     |    Pin6     | … |  Pin1   |  Pin0
        ├ CNF7 ┤ MODE7 ┤ CNF6 ┤ MODE6 ┤ … ┤ CNF1 ┤ MODE1 ┤ CNF0 ┤ MODE0
          2 bits  2 bits
```

**MODE[1:0]** — direction / vitesse :

| MODE | Signification (sortie) | Signification (entrée) |
|---|---|---|
| `00` | Entrée | Entrée |
| `01` | Sortie 10 MHz | Réservé |
| `10` | Sortie 2 MHz | Réservé |
| `11` | Sortie 50 MHz | Réservé |

**CNF[1:0]** — type de configuration :

| CNF | Mode **entrée** | Mode **sortie** |
|---|---|---|
| `00` | Analogique | Push-pull |
| `01` | Flottante | Open-drain |
| `10` | Pull-up / pull-down | AF push-pull |
| `11` | Réservé | AF open-drain |

### 🐍 Simulation Python — Calcul de CRL / CRH (10 min)

Calculons la valeur hexadécimale des registres **CRL** et **CRH** pour différentes configurations.

In [ ]:
# ============================================================
# Calcul des registres CRL / CRH pour GPIOx
# ============================================================

# Encodage des 4 bits CNF/MODE par configuration
CONFIGS = {
    # (CNF, MODE) → description
    (0b00, 0b00): "Entrée analogique",
    (0b01, 0b00): "Entrée flottante",
    (0b10, 0b00): "Entrée pull-up/down",
    (0b00, 0b11): "Sortie push-pull 50 MHz",
    (0b01, 0b11): "Sortie open-drain 50 MHz",
    (0b10, 0b11): "AF push-pull 50 MHz",
    (0b11, 0b11): "AF open-drain 50 MHz",
    (0b00, 0b10): "Sortie push-pull 2 MHz",
}

def bits_config(cnf, mode):
    """Retourne la valeur 4 bits (CNF<<2 | MODE)."""
    return ((cnf & 0x3) << 2) | (mode & 0x3)

def calcul_crl(*pins_cfg):
    """
    pins_cfg : liste de tuples (pin, cnf, mode) pour pins 0..7
    Retourne la valeur 32 bits de CRL.
    """
    crl = 0
    for pin, cnf, mode in pins_cfg:
        if not (0 <= pin <= 7):
            raise ValueError(f"Pin {pin} hors plage CRL (0-7)")
        crl |= bits_config(cnf, mode) << (pin * 4)
    return crl

def calcul_crh(*pins_cfg):
    """Comme CRL, mais pour pins 8..15."""
    crh = 0
    for pin, cnf, mode in pins_cfg:
        if not (8 <= pin <= 15):
            raise ValueError(f"Pin {pin} hors plage CRH (8-15)")
        crh |= bits_config(cnf, mode) << ((pin - 8) * 4)
    return crh

# -----------------------------------------------------------
# Exemple 1 : PA0 = entrée flottante, PA1 = sortie push-pull 50 MHz
# -----------------------------------------------------------
crl = calcul_crl(
    (0, 0b01, 0b00),   # PA0 : entrée flottante
    (1, 0b00, 0b11),   # PA1 : sortie PP 50 MHz
)
print(f"GPIOA->CRL = 0x{crl:08X}")
print(f"  Bits [3:0]   = 0b{crl & 0xF:04b}  → PA0 = entrée flottante")
print(f"  Bits [7:4]   = 0b{(crl >> 4) & 0xF:04b}  → PA1 = sortie PP 50 MHz")

# -----------------------------------------------------------
# Exemple 2 : PA8 = AF push-pull, PA13 = entrée pull-up
# -----------------------------------------------------------
crh = calcul_crh(
    (8,  0b10, 0b11),   # PA8 : AF PP 50 MHz
    (13, 0b10, 0b00),   # PA13 : entrée pull-up
)
print(f"\nGPIOA->CRH = 0x{crh:08X}")
print(f"  Bits [3:0]   = 0b{crh & 0xF:04b}  → PA8  = AF PP 50 MHz")
print(f"  Bits [23:20] = 0b{(crh >> 20) & 0xF:04b} → PA13 = entrée pull-up")

### 📖 2.5 — Registres IDR, ODR, BSRR

| Registre | Bits utilisés | Sens |
|---|---|---|
| **IDR** | 16 bits (bit i = pin i) | **Lecture seule** |
| **ODR** | 16 bits | Lecture / Écriture |
| **BSRR** | 32 bits | Écriture seule |

**BSRR** — *Bit Set/Reset Register* :
```
Bits [15:0]  : BSy  — écriture 1 → ODR.y = 1 (SET)
Bits [31:16] : BRy  — écriture 1 → ODR.y = 0 (RESET)
```

**Avantage de BSRR sur ODR** :
- **Atomique** : pas de *read-modify-write*, pas de risque de race condition.
- **Rapide** : une seule écriture pour Set ou Reset.
- **Sûr** : pas d'influence sur les autres bits du port.

### 🐍 Simulation Python — Manipulation de BSRR (5 min)

In [ ]:
# ============================================================
# Simulation d'un GPIO avec BSRR / ODR / IDR
# ============================================================

class GPIO:
    def __init__(self, nom):
        self.nom = nom
        self.odr = 0
        self.idr = 0
        self.crl = 0
        self.crh = 0

    def bsrr(self, valeur):
        """Écrit dans BSRR : bits bas = SET, bits hauts = RESET."""
        set_bits   =  valeur & 0x0000FFFF
        reset_bits = (valeur >> 16) & 0x0000FFFF
        self.odr |=  set_bits
        self.odr &= ~reset_bits & 0xFFFF

    def write(self, pin, val):
        if val:
            self.bsrr(1 << pin)
        else:
            self.bsrr(1 << (pin + 16))

    def toggle(self, pin):
        self.odr ^= (1 << pin)

    def read(self, pin):
        return (self.idr >> pin) & 1

    def __str__(self):
        bits = format(self.odr, "016b")
        return f"{self.nom}->ODR = 0b{bits}"

# Démonstration
gpioc = GPIO("GPIOC")
print("État initial :", gpioc)

gpioc.bsrr(1 << 13)         # SET PC13
print("Après SET PC13 :", gpioc)

gpioc.bsrr(1 << (13 + 16))  # RESET PC13
print("Après RESET PC13 :", gpioc)

gpioc.toggle(13)
gpioc.toggle(14)
print("Après toggle 13+14 :", gpioc)

# SET PC13 et RESET PC14 en une seule écriture atomique
gpioc.bsrr((1 << 13) | (1 << (14 + 16)))
print("Après BSRR combiné :", gpioc)

---

## 🔹 Activité 3 — Modes GPIO (25 min)

### 📖 3.1 — Entrées

| Mode | Description | Cas d'usage |
|---|---|---|
| **Analogique** | Pin isolée des buffers numériques | ADC, DAC |
| **Flottante** | Impédance haute, état indéterminé si non connectée | Signal externe avec driver |
| **Pull-up** | Résistance interne (~40 kΩ) vers VDD | Bouton actif bas |
| **Pull-down** | Résistance interne (~40 kΩ) vers VSS | Bouton actif haut |

### 📖 3.2 — Sorties

| Mode | Description | Cas d'usage |
|---|---|---|
| **Push-pull** | Sort 0 ou 1, drive fort (jusqu'à 20 mA) | LED, signal logique |
| **Open-drain** | Sort 0 ou haute impédance (nécessite pull-up externe) | Bus I2C, partage de ligne |

### 📖 3.3 — Sortie push-pull vs open-drain

```
Push-pull :                          Open-drain :
   VDD ────┐                            VDD
          │                             │
        ┌─┴─┐  P-MOS                    (pull-up externe)
  OUT ──┤   ├───                          │
        └─┬─┘                          ┌──┴──┐  N-MOS
        ┌─┴─┐  N-MOS              OUT ─┤     │
        │   │                          └──┬──┘
       VSS ─┘                            VSS
```

**Résumé :**
- **Push-pull** : 2 transistors, sortie active dans les 2 sens.
- **Open-drain** : 1 transistor, uniquement capable de tirer à 0.

### 📖 3.4 — Résistances pull-up / pull-down internes

| Paramètre | Valeur (typique) |
|---|---|
| Résistance pull-up | 30 – 50 kΩ |
| Résistance pull-down | 30 – 50 kΩ |
| Courant max par pin | ±20 mA |
| Courant total VDD/VSS | 150 mA |

> 💡 Pour un bouton, une pull-up interne suffit souvent. Pour un signal rapide, préférer une pull-up externe plus faible (1–10 kΩ).

---

## 🔹 Activité 4 — HAL vs LL vs registres (25 min)

### 📖 4.1 — Les 3 niveaux d'abstraction

| Niveau | Avantage | Inconvénient |
|---|---|---|
| **HAL** | Portable, lisible, rapide à écrire | Plus lent, plus gros |
| **LL** | Léger, proche du matériel, rapide | Moins portable |
| **Registres** | Vitesse maximale, contrôle total | Verbeux, sujet aux erreurs |

### 📖 4.2 — Exemples comparés

**HAL :**
```c
HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_SET);
HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_RESET);
HAL_GPIO_TogglePin(GPIOC, GPIO_PIN_13);
GPIO_PinState s = HAL_GPIO_ReadPin(GPIOA, GPIO_PIN_0);
```

**LL :**
```c
LL_GPIO_SetOutputPin(GPIOC, LL_GPIO_PIN_13);
LL_GPIO_ResetOutputPin(GPIOC, LL_GPIO_PIN_13);
LL_GPIO_TogglePin(GPIOC, LL_GPIO_PIN_13);
uint32_t s = LL_GPIO_IsInputPinSet(GPIOA, LL_GPIO_PIN_0);
```

**Registres (CMSIS) :**
```c
GPIOC->BSRR = GPIO_BSRR_BS13;   // Set PC13
GPIOC->BSRR = GPIO_BSRR_BR13;   // Reset PC13
GPIOC->ODR ^= (1U << 13);       // Toggle
uint16_t s = GPIOA->IDR & 0x0001U;
```

### 📖 4.3 — Comment choisir ?

| Contexte | Choix recommandé |
|---|---|
| Prototypage rapide, cours | **HAL** |
| Code de production embarqué | **LL** |
| ISR ultra-rapide, timing critique | **Registres** |
| Portage multi-plateforme | **HAL** |
| Optimisation taille / vitesse | **LL** ou **registres** |

### 🐍 Comparaison Python — Nombre de cycles CPU estimés (5 min)

In [ ]:
# ============================================================
# Estimation du coût d'une écriture GPIO selon la méthode
# ============================================================

# Ordres de grandeur (à titre pédagogique, valeurs typiques)
cout_cycles = {
    "HAL_GPIO_WritePin":   30,   # appel de fonction + vérifs
    "LL_GPIO_SetOutputPin": 5,   # inline très léger
    "GPIOC->BSRR (direct)":  1,   # écriture registre atomique
}

FREQ_MHZ = 72  # SYSCLK 72 MHz

print(f"⏱️  Estimations à {FREQ_MHZ} MHz (1 cycle = {1000/FREQ_MHZ:.3f} ns)\n")
print(f"{'Méthode':<25}{'Cycles':<10}{'Temps':<12}{'Max fréquence'}")
print("-" * 65)

for methode, cyc in cout_cycles.items():
    temps_ns = cyc * (1000 / FREQ_MHZ)
    freq_max = FREQ_MHZ * 1e6 / cyc
    if freq_max >= 1e6:
        freq_str = f"{freq_max/1e6:.2f} MHz"
    else:
        freq_str = f"{freq_max/1e3:.0f} kHz"
    print(f"{methode:<25}{cyc:<10}{temps_ns:>6.1f} ns   {freq_str}")

print("\n📌 À retenir :")
print("  → En dessous de ~100 kHz, HAL suffit largement.")
print("  → Au-delà, préférer LL ou registres.")
print("  → Dans une ISR courte, éviter HAL.")

---

## 🔹 Activité 5 — QCM formatif (10 min)

**1. Combien de broches compte un port GPIO ?**  
A. 8  
B. 16  
C. 32  
D. 64

**2. Le registre CRL configure les pins :**  
A. 0 à 7  
B. 8 à 15  
C. 0 à 15  
D. Toutes

**3. Pour faire clignoter une LED rapidement, on préfère :**  
A. `GPIOC->ODR ^= (1 << 13)`  
B. `HAL_GPIO_TogglePin(GPIOC, GPIO_PIN_13)`  
C. Les deux sont équivalents en vitesse  
D. Aucune des deux

**4. BSRR permet de :**  
A. Lire l'état des entrées  
B. Set/Reset atomique d'une pin  
C. Configurer le mode  
D. Verrouiller la config

**5. Le mode open-drain est utilisé principalement pour :**  
A. Les LED  
B. Les bus partagés (I2C)  
C. Les ADC  
D. Les timers

**6. La macro qui active l'horloge de GPIOA est :**  
A. `RCC_GPIOA_ENABLE()`  
B. `__HAL_RCC_GPIOA_CLK_ENABLE()`  
C. `HAL_GPIOA_Init()`  
D. `GPIOA_CLOCK_ON()`

### ✅ Corrigé du QCM formatif

| Q | Réponse | Explication |
|---|---|---|
| 1 | **B — 16** | Chaque GPIOx a 16 broches (0–15) |
| 2 | **A — 0 à 7** | CRL → pins 0–7, CRH → pins 8–15 |
| 3 | **A — ODR ^=** | Accès registre direct = plus rapide |
| 4 | **B — Set/Reset atomique** | BSy set, BRy reset |
| 5 | **B — Bus partagés** | Permet plusieurs maîtres sur une ligne |
| 6 | **B — `__HAL_RCC_GPIOA_CLK_ENABLE()`** | Macro HAL standard |

**Mon score : ___ / 6**

---

# 🛠️ PARTIE B — ATELIER / TP (1h30)

## 🧪 TP3 — LED + bouton avec anti-rebond

### 🎯 Objectif
Réaliser un compteur d'appuis sur bouton, affiché en binaire sur des LEDs, avec un **anti-rebond** logiciel robuste.

### 📋 Tâches à réaliser (par binôme)

| # | Tâche | Durée | Livrable |
|---|---|---|---|
| 1 | Câbler un bouton sur PA0 (pull-down externe 10 kΩ) | 10 min | Photo du montage |
| 2 | Configurer PA0 en entrée et PC13–PC15 + PA1 en sortie | 10 min | Capture CubeMX |
| 3 | Compter les appuis et les afficher sur 4 LEDs | 25 min | Code + démo |
| 4 | Implémenter un debounce logiciel (delay + relecture) | 20 min | Code commenté |
| 5 | Comparer avec un debounce matériel (RC + Schmitt) | 15 min | Schéma |
| 6 | Rédiger le compte-rendu | 10 min | CR |

### ⚙️ Code — Compteur d'appuis avec anti-rebond logiciel

In [ ]:
/* ============================================================
   TP3 - Compteur d'appuis avec anti-rebond logiciel
   Cible : STM32F103C6T6
   - Bouton   : PA0  (entrée, pull-down externe)
   - LEDs     : PC13, PC14, PC15, PA1  (sortie push-pull)
   ============================================================ */

#include "main.h"

static uint8_t compteur = 0;

/* --- Anti-rebond logiciel par relecture différée --- */
static uint8_t lire_bouton_stable(void)
{
    if (HAL_GPIO_ReadPin(GPIOA, GPIO_PIN_0) == GPIO_PIN_SET)
    {
        HAL_Delay(20);  // attendre la fin des rebonds (~20 ms)
        if (HAL_GPIO_ReadPin(GPIOA, GPIO_PIN_0) == GPIO_PIN_SET)
        {
            return 1;
        }
    }
    return 0;
}

/* --- Affichage du compteur en binaire sur 4 LEDs --- */
static void afficher_compteur(uint8_t val)
{
    HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, (val & 0x1) ? GPIO_PIN_SET : GPIO_PIN_RESET);
    HAL_GPIO_WritePin(GPIOC, GPIO_PIN_14, (val & 0x2) ? GPIO_PIN_SET : GPIO_PIN_RESET);
    HAL_GPIO_WritePin(GPIOC, GPIO_PIN_15, (val & 0x4) ? GPIO_PIN_SET : GPIO_PIN_RESET);
    HAL_GPIO_WritePin(GPIOA, GPIO_PIN_1,  (val & 0x8) ? GPIO_PIN_SET : GPIO_PIN_RESET);
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();

    afficher_compteur(compteur);

    while (1)
    {
        if (lire_bouton_stable())
        {
            compteur = (compteur + 1) & 0x0F;  // 0..15
            afficher_compteur(compteur);

            // Attendre le relâchement du bouton (front)
            while (HAL_GPIO_ReadPin(GPIOA, GPIO_PIN_0) == GPIO_PIN_SET) {}
            HAL_Delay(20);
        }
    }
}

### 🔍 Analyse du code

| Élément | Rôle |
|---|---|
| `HAL_Delay(20)` | Attend la fin des rebonds mécaniques (typ. 5–20 ms) |
| Double lecture | Confirme que le niveau est stable |
| `compteur & 0x0F` | Garde le compteur entre 0 et 15 |
| Attente du relâchement | Évite de compter plusieurs fois par appui |

### 🐍 Simulation Python — Rebonds d'un bouton (15 min)

Simulons le comportement mécanique d'un bouton pour visualiser l'effet des rebonds et tester notre anti-rebond.

In [ ]:
# ============================================================
# Simulation des rebonds d'un bouton + anti-rebond logiciel
# ============================================================

import random

random.seed(42)

def generer_signal_bouton(n_echantillons=200, periode_ms=1):
    """
    Simule un appui de bouton entre t=50ms et t=120ms,
    avec rebonds pendant 15 ms après chaque transition.
    Retourne une liste de 0/1 (1 = appuyé).
    """
    signal = [0] * n_echantillons
    t_appui   = 50
    t_relache = 120

    # Phase appui
    for t in range(t_appui, t_relache):
        signal[t] = 1

    # Rebonds à l'appui (15 premières ms)
    for t in range(t_appui, t_appui + 15):
        if random.random() < 0.5:
            signal[t] = 0

    # Rebonds au relâchement (15 ms après)
    for t in range(t_relache, t_relache + 15):
        if random.random() < 0.5:
            signal[t] = 1

    return signal

def debounce(signal, seuil_ms=20):
    """
    Anti-rebond par confirmation : ne valide un changement
    que si le niveau reste stable pendant `seuil_ms` échantillons.
    """
    sortie = [0] * len(signal)
    etat   = 0
    stable = 0
    for i, s in enumerate(signal):
        if s == etat:
            stable += 1
        else:
            stable = 0
        if stable >= seuil_ms and s != etat:
            etat = s
        sortie[i] = etat
    return sortie

def afficher_signal(signal, largeur=80):
    """Affiche le signal sous forme de blocs ASCII."""
    pas = max(1, len(signal) // largeur)
    ligne = ""
    for i in range(0, len(signal), pas):
        bloc = signal[i:i+pas]
        m = sum(bloc) / len(bloc)
        if m > 0.66: ligne += "█"
        elif m > 0.33: ligne += "▒"
        else: ligne += " "
    return ligne

brut      = generer_signal_bouton()
filtre    = debounce(brut, seuil_ms=20)

print("Signal brut (avec rebonds) :")
print("|" + afficher_signal(brut) + "|")
print("\nSignal filtré (anti-rebond) :")
print("|" + afficher_signal(filtre) + "|")

# Détection d'appuis (fronts montants)
def compter_fronts(signal):
    fronts = 0
    for i in range(1, len(signal)):
        if signal[i-1] == 0 and signal[i] == 1:
            fronts += 1
    return fronts

print(f"\n📊 Fronts montants - brut   : {compter_fronts(brut)}")
print(f"📊 Fronts montants - filtré : {compter_fronts(filtre)}")

### 🐍 Anti-rebond par fenêtre temporelle (variante)

Version alternative utilisée dans un contexte **temps réel** (pas de `HAL_Delay`).

In [ ]:
/* ============================================================
   Anti-rebond non bloquant (basé sur HAL_GetTick)
   ============================================================ */

#define DEBOUNCE_MS  20

static uint8_t dernier_etat    = 0;
static uint32_t dernier_tick   = 0;

uint8_t bouton_appuye(void)
{
    uint8_t etat = HAL_GPIO_ReadPin(GPIOA, GPIO_PIN_0);
    if (etat != dernier_etat && (HAL_GetTick() - dernier_tick) > DEBOUNCE_MS)
    {
        dernier_tick = HAL_GetTick();
        dernier_etat = etat;
        return etat;  // 1 = vient d'être appuyé
    }
    return 0;
}

### 📝 Compte-rendu de TP3

**Nom :** __________________  **Prénom :** __________________  **Binôme :** __________________

**1. Configuration CubeMX**
- PA0 : mode ... , pull ...
- PC13 / PC14 / PC15 / PA1 : mode ...
- Vitesse (GPIO output speed) : ...

**2. Code ajouté dans `main.c`**
```c
// Colle ici ton code
```

**3. Observation sans anti-rebond**
- Comportement : ...
- Nombre d'incréments par appui : ...

**4. Observation avec anti-rebond (20 ms)**
- Comportement : ...
- Nombre d'incréments par appui : ...

**5. Comparaison avec debounce matériel (RC + Schmitt)**
- Schéma : ...
- Avantages / inconvénients : ...

**6. Problèmes rencontrés**
- ...

**7. Solutions apportées**
- ...

### 🧪 Exercice bonus — Chenillard 4 LED avec 2 boutons

Ajouter un second bouton sur **PA2** qui inverse le sens du chenillard.

**Cahier des charges :**
- Sans appui : chenillard PC13 → PC14 → PC15 → PA1 → PC13 …
- Après appui sur PA2 : sens inverse
- Fréquence : 250 ms par étape
- Utiliser un anti-rebond sur PA2

In [ ]:
// Squelette solution bonus

#define DEBOUNCE_MS 20

static const uint16_t leds[4] = {
    /* port, pin */
};

static int8_t sens = 1;   // 1 = avant, -1 = arrière
static int8_t etape = 0;

/* À compléter :
   - initialiser sens avec un debounce sur PA2
   - déplacer la LED active selon sens et etape
   - appeler HAL_Delay(250)
*/

---

# 🏠 PARTIE C — HOMEWORK (1h30)

## 📚 Exercices à rendre

### 🧩 Exercice 1 — Calcul de CRL / CRH (30 min)

Pour chaque configuration, calcule la valeur hexadécimale de **CRL** et **CRH**.

**Config A — Port A :**
- PA0 : entrée analogique
- PA1 : entrée flottante
- PA2 : entrée pull-up
- PA3 : sortie push-pull 2 MHz
- PA4 : sortie push-pull 50 MHz
- PA5 : AF push-pull 50 MHz
- PA6 : AF open-drain 50 MHz
- PA7 : sortie open-drain 2 MHz
- PA8 : entrée pull-down
- PA9 : AF push-pull 50 MHz
- PA10 : entrée flottante
- PA13 : entrée pull-up (SWDIO)
- PA14 : entrée pull-down (SWCLK)

👉 Réponse : `GPIOA->CRL = 0x________` et `GPIOA->CRH = 0x________`

In [ ]:
# Corrigé Exercice 1

crl_A = calcul_crl(
    (0, 0b00, 0b00),   # PA0 entrée analogique
    (1, 0b01, 0b00),   # PA1 entrée flottante
    (2, 0b10, 0b00),   # PA2 entrée pull-up
    (3, 0b00, 0b10),   # PA3 sortie PP 2 MHz
    (4, 0b00, 0b11),   # PA4 sortie PP 50 MHz
    (5, 0b10, 0b11),   # PA5 AF PP 50 MHz
    (6, 0b11, 0b11),   # PA6 AF OD 50 MHz
    (7, 0b01, 0b10),   # PA7 sortie OD 2 MHz
)

crh_A = calcul_crh(
    (8,  0b10, 0b00),  # PA8  entrée pull-down
    (9,  0b10, 0b11),  # PA9  AF PP 50 MHz
    (10, 0b01, 0b00),  # PA10 entrée flottante
    (13, 0b10, 0b00),  # PA13 entrée pull-up
    (14, 0b10, 0b00),  # PA14 entrée pull-down (idem pull-up/down via ODR)
)

print(f"GPIOA->CRL = 0x{crl_A:08X}")
print(f"GPIOA->CRH = 0x{crh_A:08X}")

### 🧩 Exercice 2 — HAL vs LL vs registres (30 min)

Réécrire **la même fonction** dans les 3 styles :

**Objectif :** faire clignoter PC13 avec un cycle ON = 100 ms, OFF = 900 ms.

1. Version **HAL**
2. Version **LL**
3. Version **registres** (CMSIS)

Indiquer pour chaque version :
- Nombre de lignes
- Inclusions nécessaires
- Avantages / inconvénients

In [ ]:
/* ============ VERSION HAL ============ */
void blink_hal(void)
{
    HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_SET);
    HAL_Delay(100);
    HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_RESET);
    HAL_Delay(900);
}

/* ============ VERSION LL ============ */
void blink_ll(void)
{
    LL_GPIO_SetOutputPin(GPIOC, LL_GPIO_PIN_13);
    HAL_Delay(100);
    LL_GPIO_ResetOutputPin(GPIOC, LL_GPIO_PIN_13);
    HAL_Delay(900);
}

/* ============ VERSION REGISTRES ============ */
void blink_reg(void)
{
    GPIOC->BSRR = GPIO_BSRR_BS13;   // Set PC13
    HAL_Delay(100);
    GPIOC->BSRR = GPIO_BSRR_BR13;   // Reset PC13
    HAL_Delay(900);
}

**✍️ Comparaison personnelle :**

| Critère | HAL | LL | Registres |
|---|---|---|---|
| Nombre de lignes | ... | ... | ... |
| Inclusions | ... | ... | ... |
| Lisibilité | ... | ... | ... |
| Vitesse | ... | ... | ... |
| Portabilité | ... | ... | ... |

### 🧩 Exercice 3 — Lecture du RM0008 (30 min)

Lire le **chapitre 9 (GPIO)** du RM0008 et répondre :

1. Quels sont les 3 modes d'entrée possibles (hors analogique) ?
2. Quelle est la différence entre le mode « entrée pull-up » et « entrée pull-down » ? Comment les distingue-t-on dans le registre ?
3. Que se passe-t-il si on écrit `1` puis `0` dans le même cycle sur BSy et BRy ?
4. Pourquoi `BRR` est-il déprécié en faveur de `BSRR` ?
5. Sur la Blue Pill, pourquoi **PC14** et **PC15** sont-elles à éviter pour du GPIO pur ?

### ✍️ Réponses

1. ...
2. ...
3. ...
4. ...
5. ...

---

## 🧮 Exercice supplémentaire — Mini simulateur GPIO (optionnel)

Complète la classe `GPIOSim` pour qu'elle supporte :
- `configurer(pin, mode)` où `mode` ∈ {`'IN_FLT'`, `'IN_PU'`, `'IN_PD'`, `'IN_AN'`, `'OUT_PP'`, `'OUT_OD'`, `'AF_PP'`, `'AF_OD'`}
- `ecrire(pin, val)` : refuse si le pin n'est pas une sortie
- `lire(pin)` : refuse si le pin est une sortie
- `afficher()` : tableau récapitulatif des pins

In [ ]:
# À compléter
class GPIOSim:
    MODES = {
        'IN_FLT': (0b01, 0b00),
        'IN_PU':  (0b10, 0b00),
        'IN_PD':  (0b10, 0b00),
        'IN_AN':  (0b00, 0b00),
        'OUT_PP': (0b00, 0b11),
        'OUT_OD': (0b01, 0b11),
        'AF_PP':  (0b10, 0b11),
        'AF_OD':  (0b11, 0b11),
    }

    def __init__(self, nom):
        self.nom = nom
        self.pins = {}   # pin -> mode
        self.odr = 0

    def configurer(self, pin, mode):
        # TODO
        pass

    def ecrire(self, pin, val):
        # TODO
        pass

    def lire(self, pin):
        # TODO
        pass

    def afficher(self):
        # TODO
        pass

# Test attendu
# g = GPIOSim("GPIOC")
# g.configurer(13, 'OUT_PP')
# g.ecrire(13, 1)
# g.afficher()

In [ ]:
# ✅ Corrigé
class GPIOSimOK:
    MODES = {
        'IN_FLT': (0b01, 0b00),
        'IN_PU':  (0b10, 0b00),
        'IN_PD':  (0b10, 0b00),
        'IN_AN':  (0b00, 0b00),
        'OUT_PP': (0b00, 0b11),
        'OUT_OD': (0b01, 0b11),
        'AF_PP':  (0b10, 0b11),
        'AF_OD':  (0b11, 0b11),
    }

    def __init__(self, nom):
        self.nom = nom
        self.pins = {}
        self.odr = 0

    def configurer(self, pin, mode):
        if mode not in self.MODES:
            raise ValueError(f"Mode inconnu : {mode}")
        if not (0 <= pin <= 15):
            raise ValueError("Pin hors plage")
        self.pins[pin] = mode

    def ecrire(self, pin, val):
        mode = self.pins.get(pin)
        if mode is None or not mode.startswith('OUT'):
            raise ValueError(f"Pin {pin} n'est pas une sortie (mode={mode})")
        if val:
            self.odr |= (1 << pin)
        else:
            self.odr &= ~(1 << pin)

    def lire(self, pin):
        mode = self.pins.get(pin)
        if mode is None or mode.startswith('OUT'):
            raise ValueError(f"Pin {pin} n'est pas une entrée (mode={mode})")
        return (self.odr >> pin) & 1

    def afficher(self):
        print(f"=== {self.nom} ===")
        for p in range(16):
            m = self.pins.get(p, '-')
            bit = (self.odr >> p) & 1 if m.startswith('OUT') else '-'
            print(f"  P{p:<2} : {m:<8} ODR[{p}]={bit}")

g = GPIOSimOK("GPIOC")
g.configurer(13, 'OUT_PP')
g.configurer(14, 'OUT_PP')
g.configurer(15, 'IN_PU')
g.ecrire(13, 1)
g.ecrire(14, 0)
g.afficher()

---
# ✅ PARTIE D — AUTO-ÉVALUATION Semaine 3

Coche ce que tu maîtrises.

- [ ] Je connais les ports GPIO disponibles sur le STM32F103C6T6.
- [ ] Je connais l'adresse de base de GPIOA / GPIOB / GPIOC.
- [ ] Je sais lire et écrire les registres CRL / CRH.
- [ ] Je sais utiliser IDR / ODR / BSRR.
- [ ] Je comprends la différence entre push-pull et open-drain.
- [ ] Je sais choisir pull-up vs pull-down pour un bouton.
- [ ] Je sais écrire un code GPIO en HAL, LL et registres.
- [ ] Je comprends l'avantage atomique du BSRR.
- [ ] J'ai implémenté un anti-rebond logiciel.
- [ ] J'ai testé les rebonds à l'oscilloscope ou en simulation.
- [ ] J'ai activé l'horloge GPIO avec `__HAL_RCC_xxx_CLK_ENABLE()`.
- [ ] J'ai rédigé mon compte-rendu de TP3.
- [ ] J'ai complété le calcul des CRL / CRH.
- [ ] J'ai lu le chapitre 9 du RM0008.

### 📊 Mon score : ___ / 14

| Score | Interprétation |
|---|---|
| 12–14 | ✅ Prêt pour la S4 (EXTI/NVIC) |
| 8–11 | ⚠️ Revoir les points manquants |
| < 8 | 🔁 Reprendre les activités 2 à 5 |

---
# 📚 RESSOURCES Semaine 3

### Documents officiels
- 📄 **RM0008** — chapitre 9 (General-purpose and alternate-function I/Os)
- 📄 **Datasheet STM32F103x6** — section 5.3 (Pinouts and pin description)
- 📄 **PM0056** — chapitre sur les instructions atomiques (BSRR-like)

### Outils
- **STM32CubeMX** — onglet *Pinout & Configuration* → GPIO
- **Oscilloscope** ou **analyseur logique** pour visualiser les rebonds
- **STM32CubeProgrammer** pour lire les registres GPIO en live

### Vidéos
- *STM32 GPIO Tutorial (HAL + Register)* — ControllersTech
- *Button Debouncing Techniques* — YouTube

---

### 🔗 Passage à la semaine 4

**Prochaine séance :** EXTI + NVIC  
- Principe des interruptions
- Lignes EXTI0–EXTI15 et AFIO_EXTICR
- Priorités NVIC et préemption
- Écriture d'une ISR avec HAL

**Préparation :** Lire le chapitre 10 (EXTI) du RM0008.

---

**Fin du notebook — Semaine 3** ✨